# Supplementary Results 14.1-14.2 — The genetic correlation matrix and therapeutic-area independence

What the matrix covers, and whether diseases within a therapeutic area are more genetically
correlated than diseases from two different areas — the assumption behind counting therapeutic areas
as a conservative measure of pleiotropy.

Numbers are written to `results/sr14_genetic_correlation.json`, and the therapeutic-area matrix to
`chapters/06-supplementary-tables/sheets/ST17_ta_correlation_matrix.csv`.

**Provenance.** `chapters/_legacy/06-review-r1/ta-independence/01_within_vs_between_ta.ipynb`: the
same single-area assignment by ontology descent, the same absolute correlations, and the same
permutation of area labels over diseases rather than over pairs. The LD score regression itself is
external; `01-data-preparation/13_genetic_correlation.ipynb` builds the matrix from its output.

**Deviation from the published section.** The area assignment now uses the Supplementary Table 9 root
ordering, not the legacy ordering the published section was built on. Thirteen of the numbers in
Supplementary Results 14.2 move in the third decimal or in a pair count; the direction, the fold and
the permutation P are unchanged. The chapter README lists them.

**Not yet ported.** Supplementary Results 14.3 (effective independent traits) and 14.4 (disease-list
subsampling) remain in `chapters/_legacy/06-review-r1/effective-independent-traits/` and
`disease-subsampling/`. See the chapter README.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

from manuscript_methods import paper

numbers = {}
SEED = 20260813
PERMUTATIONS = 10000

matrix = pd.read_parquet(paper.derived("rg_matrix"))
print(f"matrix: {matrix.shape[0]} traits")

matrix: 1114 traits


## What the matrix holds and covers

In [2]:
# A trait counts as a measurement when the **upstream** `therapeutic_area` column of the canonical
# pairwise table says so. That is the labelling the published analysis used
# (`_legacy/06-review-r1/ta-independence/01_within_vs_between_ta.ipynb`, cell 3), and it reproduces
# 551 diseases and 563 measurements exactly. It is not the same as the trait's own ontology
# classification: 283 of the 1,114 traits carry no upstream label at all and count as diseases here,
# while `efo_therapeutic_area.primaryTherapeuticArea` puts them under the measurement root, giving
# 532/582. The matrix itself is built from this same file, so its labels travel with it.
pairwise = pd.read_parquet(
    paper.baseline("canonical_pairwise_table") + "/canonical_pairwise_table.parquet",
    columns=["diseaseId_1", "therapeutic_area_1", "diseaseId_2", "therapeutic_area_2"],
)
upstream = (
    pd.concat(
        [
            pairwise[["diseaseId_1", "therapeutic_area_1"]].set_axis(["trait", "area"], axis=1),
            pairwise[["diseaseId_2", "therapeutic_area_2"]].set_axis(["trait", "area"], axis=1),
        ]
    )
    .drop_duplicates("trait")
    .set_index("trait")["area"]
)
is_measurement = upstream.reindex(matrix.index).eq("measurement")

# The ontology-based alternative, for the record.
areas = pd.read_parquet(paper.derived("efo_therapeutic_area")).set_index("id")["primaryTherapeuticArea"]
by_term = areas.reindex(matrix.index) == paper.MEASUREMENT

numbers["S14.01"] = len(matrix)
numbers["S14.03"] = int(is_measurement.sum())
numbers["S14.02"] = len(matrix) - numbers["S14.03"]

off_diagonal = ~np.eye(len(matrix), dtype=bool)
measured = (matrix.to_numpy() != 0) & off_diagonal
numbers["S14.04"] = round(100 * measured.sum() / off_diagonal.sum(), 2)
print(f"traits {numbers['S14.01']} | diseases {numbers['S14.02']} | measurements {numbers['S14.03']}")
print(f"traits with no upstream label, counted as diseases: {int(upstream.reindex(matrix.index).isna().sum())}")
print(f"the term-level rule would give: diseases {int((~by_term).sum())} | measurements {int(by_term.sum())}")
print(f"off-diagonal entries with a measured correlation: {numbers['S14.04']}%")

traits 1114 | diseases 551 | measurements 563
traits with no upstream label, counted as diseases: 283
the term-level rule would give: diseases 532 | measurements 582
off-diagonal entries with a measured correlation: 99.84%


In [3]:
diseases = pd.read_parquet(paper.derived("prioritised_genes_diseases"), columns=["geneId", "diseaseIds"])
measurements = pd.read_parquet(paper.derived("prioritised_genes_measurements"), columns=["geneId", "diseaseIds"])
# The published association counts are over the genes of `gene_table`, the protein-coding set every
# gene-level analysis in this work uses: 115,017 gene-measurement associations, not the 150,360 the
# unrestricted prioritisation table holds. The disease side is unaffected, since all of its genes are
# already in that table. The *term* counts stay unrestricted, which is what the published table's own
# definition says ("unique disease or measurement ontology terms used in this work").
gene_table_genes = set(pd.read_parquet(paper.derived("gene_table"), columns=["geneId"])["geneId"])


def coverage(frame, label):
    """Terms and gene-trait associations, and how many of each the matrix covers."""
    exploded = frame.explode("diseaseIds").dropna(subset=["diseaseIds"]).drop_duplicates()
    terms = exploded["diseaseIds"].unique()
    exploded = exploded[exploded["geneId"].isin(gene_table_genes)]
    in_matrix = set(matrix.index)
    covered_terms = [t for t in terms if t in in_matrix]
    covered_rows = exploded[exploded["diseaseIds"].isin(in_matrix)]
    return [
        {"stratum": f"{label} terms", "total": len(terms), "in matrix": len(covered_terms)},
        {"stratum": f"gene-{label} associations", "total": len(exploded), "in matrix": len(covered_rows)},
    ]


coverage_table = pd.DataFrame(coverage(diseases, "disease") + coverage(measurements, "measurement"))
coverage_table["%"] = (100 * coverage_table["in matrix"] / coverage_table["total"]).round(1)

numbers["S14.05"] = int(coverage_table.loc[0, "in matrix"])
numbers["S14.06"] = float(coverage_table.loc[0, "%"])
numbers["S14.09"] = int(coverage_table.loc[1, "in matrix"])
numbers["S14.10"] = float(coverage_table.loc[1, "%"])
numbers["S14.07"] = int(coverage_table.loc[2, "in matrix"])
numbers["S14.08"] = float(coverage_table.loc[2, "%"])
numbers["S14.11"] = int(coverage_table.loc[3, "in matrix"])
numbers["S14.12"] = float(coverage_table.loc[3, "%"])
coverage_table

,stratum,total,in matrix,%
0,disease terms,1394,471,33.8
1,gene-disease associations,36858,27006,73.3
2,measurement terms,3412,507,14.9
3,gene-measurement associations,115017,86522,75.2


## Assigning each disease trait to a single therapeutic area

The first area whose ontology subtree contains the term, in the Supplementary Table 9 ordering of the
23 roots — the same ordering `primaryTherapeuticArea` uses and the one the rest of the pipeline was
unified on. Terms under no therapeutic-area root — mostly phenotype codes sitting under a general
phenotype term — are left out of the comparison.

In [4]:
ontology = pd.read_parquet(paper.release("disease") + "/disease.parquet", columns=["id", "descendants", "ancestors"])
roots = ontology[ontology["id"].isin(paper.THERAPEUTIC_AREAS)]
descendants = {row.id: set(list(row.descendants) if row.descendants is not None else []) for row in roots.itertuples()}
# The Supplementary Table 9 ordering, which is what `paper.THERAPEUTIC_AREAS` and the
# `primaryTherapeuticArea` column carry. The published section used the legacy ordering instead,
# which put `genetic, familial or congenital disease` second to last and `immune system disease`
# eighth rather than seventeenth; this section was the last place still on it. Switching reassigns
# 41 of the 400 diseases, gives 22 areas instead of 21, and moves a net 55 pairs from within-area to
# between-area. See `01-data-preparation/03_therapeutic_areas` and the chapter README.
priority = [key for key in paper.THERAPEUTIC_AREAS if key != paper.MEASUREMENT]


def single_area(term):
    """First therapeutic area in the Supplementary Table 9 order whose subtree holds this term."""
    holding = {root for root, kids in descendants.items() if term == root or term in kids}
    for root in priority:
        if root in holding:
            return root
    return "other"


disease_traits = [t for t in matrix.index if not bool(is_measurement.get(t, False))]
area_of = {t: single_area(t) for t in disease_traits}
analysis_traits = [t for t in disease_traits if area_of[t] != "other"]

numbers["S14.13"] = len(analysis_traits)
numbers["S14.14"] = len(disease_traits) - len(analysis_traits)
numbers["S14.15"] = len({area_of[t] for t in analysis_traits})
print(
    f"disease traits in the matrix: {len(disease_traits)} | carrying an area: {numbers['S14.13']} | "
    f"under no area root: {numbers['S14.14']} | areas represented: {numbers['S14.15']}"
)

# `single_area` counts a root as its own area (`term == root`), which the derived
# `primaryTherapeuticArea` column cannot do because a term is not its own ancestor. That is the only
# difference between the two: 10 of the 400 analysis traits are therapeutic-area roots themselves,
# and the column calls 9 of them `other` and puts `pancreas disease` under `endocrine system
# disease`. The published trait set keeps them, so the descent rule stays.
column = pd.read_parquet(paper.derived("efo_therapeutic_area")).set_index("id")["primaryTherapeuticArea"]
disagree = [t for t in analysis_traits if area_of[t] != column.get(t)]
assert set(disagree) <= set(paper.THERAPEUTIC_AREAS), "the two rules differ on a non-root term"
print(f"analysis traits where the descent rule and primaryTherapeuticArea differ: {len(disagree)}, all roots")

disease traits in the matrix: 551 | carrying an area: 400 | under no area root: 151 | areas represented: 22
analysis traits where the descent rule and primaryTherapeuticArea differ: 10, all roots


## Within-area against between-area correlation

In [5]:
index = {t: i for i, t in enumerate(matrix.index)}
rows = np.array([index[t] for t in analysis_traits])
sub = matrix.to_numpy()[np.ix_(rows, rows)]
labels = np.array([area_of[t] for t in analysis_traits])
upper_i, upper_j = np.triu_indices(len(analysis_traits), 1)
correlation = sub[upper_i, upper_j]
same_area = labels[upper_i] == labels[upper_j]

# Pairs where one disease is a broader form of the other are near-identical by construction.
ancestors = {
    row.id: set(list(row.ancestors) if row.ancestors is not None else [])
    for row in ontology[ontology["id"].isin(analysis_traits)].itertuples()
}
terms = np.array(analysis_traits)
nested = np.array(
    [
        terms[j] in ancestors.get(terms[i], set()) or terms[i] in ancestors.get(terms[j], set())
        for i, j in zip(upper_i, upper_j)
    ]
)
print(
    f"disease pairs: {len(correlation):,} | within-area {int(same_area.sum()):,} | "
    f"between-area {int((~same_area).sum()):,} | nested {int(nested.sum()):,}"
)

disease pairs: 79,800 | within-area 5,789 | between-area 74,011 | nested 1,007


In [6]:
def compare(keep, label, draws=PERMUTATIONS, seed=SEED):
    """Within versus between comparison on one subset of pairs, with a label-permutation P value."""
    values = np.abs(correlation[keep])
    group = same_area[keep]
    within, between = values[group], values[~group]
    observed = float(within.mean() - between.mean())
    superiority = float(stats.mannwhitneyu(within, between, alternative="greater").statistic) / (
        len(within) * len(between)
    )
    fold = float((within >= 0.5).mean() / (between >= 0.5).mean())

    generator = np.random.default_rng(seed)
    extreme = 0
    for _ in range(draws):
        shuffled = generator.permutation(labels)
        permuted = (shuffled[upper_i] == shuffled[upper_j])[keep]
        if permuted.sum() == 0 or (~permuted).sum() == 0:
            continue
        if abs(values[permuted].mean() - values[~permuted].mean()) >= abs(observed):
            extreme += 1
    return {
        "pair set": label,
        "n_within": len(within),
        "n_between": len(between),
        "within": round(float(within.mean()), 3),
        "between": round(float(between.mean()), 3),
        "difference": round(observed, 3),
        ">=0.5 fold": round(fold, 2),
        "superiority": round(superiority, 3),
        "P": (extreme + 1) / (draws + 1),
    }


everything = np.ones(len(correlation), dtype=bool)
comparison = pd.DataFrame([compare(everything, "All disease pairs"), compare(~nested, "Excluding nested pairs")])
comparison

,pair set,n_within,n_between,within,between,difference,>=0.5 fold,superiority,P
0,All disease pairs,5789,74011,0.400,0.317,0.083,1.47,0.571,0.0001
1,Excluding nested pairs,5095,73698,0.387,0.317,0.071,1.41,0.559,0.0001


In [7]:
numbers["S14.16"] = int(comparison.loc[0, "n_within"])
numbers["S14.17"] = int(comparison.loc[0, "n_between"])
numbers["S14.18"] = float(comparison.loc[0, "within"])
numbers["S14.19"] = float(comparison.loc[0, "between"])
numbers["S14.20"] = float(comparison.loc[0, "difference"])
numbers["S14.21"] = float(comparison.loc[0, ">=0.5 fold"])
numbers["S14.22"] = float(comparison.loc[0, "superiority"])
numbers["S14.23"] = int(nested.sum())
numbers["S14.24"] = float(comparison.loc[1, "within"])
numbers["S14.25"] = float(comparison.loc[1, "between"])
numbers["S14.26"] = float(comparison.loc[1, ">=0.5 fold"])
numbers["S14.27"] = float(comparison.loc[1, "superiority"])
numbers["S14.28"] = int(comparison.loc[1, "n_within"])
numbers["S14.29"] = int(comparison.loc[1, "n_between"])
numbers["S14.30"] = round(100 * float((np.abs(correlation[~same_area]) >= 0.5).mean()), 0)
print(f"between-area pairs above 0.5: {numbers['S14.30']}%")
print({k: numbers[k] for k in ["S14.18", "S14.19", "S14.20", "S14.21", "S14.22"]})

between-area pairs above 0.5: 22.0%
{'S14.18': 0.4, 'S14.19': 0.317, 'S14.20': 0.083, 'S14.21': 1.47, 'S14.22': 0.571}


## Supplementary Table 17 — mean absolute correlation per therapeutic-area pair

The same pairs and the same absolute correlations, aggregated by unordered area pair. The diagonal is
the within-area comparison broken down by area and the off-diagonal the between-area one, so the
pair-count weighted means of the two halves have to return the table's within and between values.

In [8]:
import itertools

names = paper.THERAPEUTIC_AREAS
name_i = np.array([names[a] for a in labels[upper_i]])
name_j = np.array([names[a] for a in labels[upper_j]])
lower = name_i < name_j
pairs = pd.DataFrame(
    {
        "therapeuticArea1": np.where(lower, name_i, name_j),
        "therapeuticArea2": np.where(lower, name_j, name_i),
        "absRg": np.abs(correlation),
    }
)
grouped = pairs.groupby(["therapeuticArea1", "therapeuticArea2"])["absRg"].agg(["mean", "size"]).reset_index()

# Every area pair gets a row, including the ones no disease pair falls in.
present = sorted(names[a] for a in set(labels))
cells = pd.DataFrame(
    [{"therapeuticArea1": a, "therapeuticArea2": b} for a, b in itertools.combinations_with_replacement(present, 2)]
)
st17 = cells.merge(grouped, on=["therapeuticArea1", "therapeuticArea2"], how="left").rename(
    columns={"mean": "meanAbsRg", "size": "nPairs"}
)
st17["nPairs"] = st17["nPairs"].fillna(0).astype(int)
st17["meanAbsRg"] = st17["meanAbsRg"].round(4)
st17 = st17.sort_values(["therapeuticArea1", "therapeuticArea2"]).reset_index(drop=True)

sheet = paper.ROOT / "chapters/06-supplementary-tables/sheets/ST17_ta_correlation_matrix.csv"
st17.to_csv(sheet, index=False)

diagonal = st17[st17["therapeuticArea1"] == st17["therapeuticArea2"]]
off = st17[st17["therapeuticArea1"] != st17["therapeuticArea2"]]


def weighted(part):
    """Pair-count weighted mean of a half of the table."""
    return float((part["meanAbsRg"] * part["nPairs"]).sum() / part["nPairs"].sum())


assert diagonal["nPairs"].sum() == numbers["S14.16"], "diagonal does not hold the within-area pairs"
assert off["nPairs"].sum() == numbers["S14.17"], "off-diagonal does not hold the between-area pairs"
# The cell means carry four decimals, so the weighted means agree only to within that rounding.
assert abs(weighted(diagonal) - numbers["S14.18"]) < 0.001, "diagonal does not recover the within-area mean"
assert abs(weighted(off) - numbers["S14.19"]) < 0.001, "off-diagonal does not recover the between-area mean"

print(f"wrote {sheet.name}: {len(st17)} cells | {len(diagonal)} diagonal | {len(off)} off-diagonal")
print(f"diagonal weighted mean {weighted(diagonal):.4f} | off-diagonal {weighted(off):.4f}")
print(f"nPairs {st17['nPairs'].min()}-{st17['nPairs'].max()} | below 10 pairs {int((st17['nPairs'] < 10).sum())}")
st17.head()

wrote ST17_ta_correlation_matrix.csv: 253 cells | 22 diagonal | 231 off-diagonal
diagonal weighted mean 0.3995 | off-diagonal 0.3168
nPairs 1-2478 | below 10 pairs 9


,therapeuticArea1,therapeuticArea2,meanAbsRg,nPairs
0,cancer or benign tumor,cancer or benign tumor,0.3291,1711
1,cancer or benign tumor,cardiovascular disease,0.2303,2478
2,cancer or benign tumor,disorder of ear,0.3395,118
3,cancer or benign tumor,disorder of visual system,0.2722,885
4,cancer or benign tumor,endocrine system disease,0.2699,413


## Write the results

In [9]:
print(paper.save_results("sr14_genetic_correlation", numbers))
pd.Series(numbers).to_frame("computed")

/Users/yt4/Projects/Gentropy-manuscript/results/sr14_genetic_correlation.json


,computed
S14.01,1114.000
S14.03,563.000
S14.02,551.000
S14.04,99.840
S14.05,471.000
S14.06,33.800
S14.09,27006.000
S14.10,73.300
S14.07,507.000
S14.08,14.900
